### Digit logit analysis for CIF cell parameters

In [ ]:
CKPT = "model_ckpts/mpdb_slme/PKV-opt/checkpoint-6250"
DEVICE = "cuda:0"
CONDITION_VALUE = 1.0

### Load model

In [ ]:
from _utils._notebook_utils.y_logits_utils import load_model_for_logits

model, tokenizer = load_model_for_logits(CKPT, device=DEVICE)

### Get CIF and reconstruct bracket format
Use the exact CsSnTe2 I4/mcm CIF the model generated (SLME=28%, tetragonal a=b)

In [ ]:
import pandas as pd
from _utils._notebook_utils.y_logits_utils import reconstruct_bracketed_cif

df = pd.read_parquet("_artifacts/slme/slme-PKV-opt_gen.parquet")
mask = df["Generated CIF"].str.contains("'Rb8 Nb12 Br36'", na=False) & df["Generated CIF"].str.contains("C2/m", na=False)
print(f"Found {mask.sum()} matching entries in the DataFrame.")
raw_cif = df[mask].iloc[0]["Generated CIF"]
print(raw_cif)
print("\n")

bracketed_cif = reconstruct_bracketed_cif(raw_cif)
print(bracketed_cif)

### Extract logits

In [ ]:
from _utils._notebook_utils.y_logits_utils import extract_digit_logits

result = extract_digit_logits(
    model, tokenizer, bracketed_cif,
    condition_value=CONDITION_VALUE,
    device=DEVICE,
    include_coords=True,
    max_coord_atoms=3,
)

for f in result["fields"]:
    print(f"{f['tag']}: {''.join(f['digit_tokens'])}")

### Plot heatmaps
Cell params only (cell lengths + angles)

In [ ]:
import __init__
from _utils._notebook_utils.y_logits_utils import plot_digit_heatmap_grid

result["fields"] = [f for f in result["fields"] if f["tag"] not in ["_cell_volume", "_cell_formula_units_Z"] + [f"_atom_site_fract_x_{i}" for i in range(2, 9)]]

fig, axes = plot_digit_heatmap_grid(
    result["fields"],
    ncols=3,
    title="Rb2(NbBr3)3_gen",
    save_path="plots/Rb2(NbBr3)3_gen_tags_logits.png",
)